# Day 15: FDE Professional Skills 1
**FDE identity and mindset, and a bias towards action, through simulations and role plays**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## The 30 Minute Prototype
Today is about mindset, but FDEs act through code. This notebook supports **Activity 3**: answering "Why are our customers unhappy?" from messy support data, fast.

Rule: **good enough and shown beats perfect and hidden.** Set a 30 minute timer.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# pandas and matplotlib are already installed in Colab.

# ===== Option B: VS Code / local =====
# pip install pandas matplotlib   (in your terminal)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("Ready. Start your 30 minute timer now.")

## 2. The customer's data
We generate realistic support tickets. In real life this would be an export from their ticketing tool.

In [ ]:
rng = np.random.default_rng(15)
n = 400
cats = ["Login issues", "Billing", "Slow app", "Data export", "Feature request", "Mobile crash"]
weights = [0.22, 0.18, 0.2, 0.12, 0.18, 0.1]
category = rng.choice(cats, size=n, p=weights)
base_hours = {"Login issues": 6, "Billing": 30, "Slow app": 48, "Data export": 20, "Feature request": 72, "Mobile crash": 26}
tickets = pd.DataFrame({
    "ticket_id": range(5001, 5001 + n),
    "created": pd.Timestamp("2026-10-01") + pd.to_timedelta(rng.integers(0, 42, n), unit="D"),
    "category": category,
    "priority": rng.choice(["low", "medium", "high"], size=n, p=[0.4, 0.4, 0.2]),
    "hours_to_resolve": [max(1, rng.normal(base_hours[c], base_hours[c] * 0.4)) for c in category],
    "satisfaction": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.15, 0.2, 0.25, 0.25, 0.15]),
})
# Real data is messy: a few blanks and a typo
tickets.loc[rng.choice(n, 12, replace=False), "satisfaction"] = np.nan
tickets.loc[rng.choice(n, 5, replace=False), "category"] = "slow app"
tickets.head()

## 3. Agree the question and write assumptions first

In [ ]:
question = "Which issues make customers unhappy, and which should we fix first?"
assumptions = [
    {"assumption": "Satisfaction scores reflect the ticket experience", "confidence": "medium", "test": "Ask 3 support agents"},
    {"assumption": "Categories are labelled consistently", "confidence": "low", "test": "Check unique values now"},
    {"assumption": "Six weeks of data is representative", "confidence": "medium", "test": "Compare with last quarter later"},
]
print(question)
pd.DataFrame(assumptions)

## 4. Quick clean (test the low-confidence assumption immediately)

In [ ]:
print(tickets["category"].value_counts())
tickets["category"] = tickets["category"].str.strip().str.capitalize()   # "slow app" -> "Slow app"
print("\nAfter fix:", tickets["category"].nunique(), "categories")
print("Missing satisfaction:", tickets["satisfaction"].isna().sum(), "(kept, excluded from averages)")

## 5. One table that answers the question

In [ ]:
summary = (tickets.groupby("category")
           .agg(tickets=("ticket_id", "count"),
                avg_hours=("hours_to_resolve", "mean"),
                avg_satisfaction=("satisfaction", "mean"),
                high_priority=("priority", lambda s: (s == "high").sum()))
           .round(1)
           .sort_values("avg_satisfaction"))
summary["pain_score"] = (summary["tickets"] * summary["avg_hours"] / summary["avg_satisfaction"]).round(0)
summary.sort_values("pain_score", ascending=False)

## 6. One chart with a title that states the finding

In [ ]:
top = summary.sort_values("pain_score", ascending=True)
worst = list(top.index[-2:])[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top.index, top["pain_score"], color=["#0FA3A3"] * (len(top) - 2) + ["#FF8A3D"] * 2)
ax.set_title(f"{worst[0]} and {worst[1]} cause the most customer pain")   # the title states the finding
ax.set_xlabel("Pain score (tickets x hours to resolve / satisfaction)")
plt.tight_layout()
plt.show()

## 7. Three sentences of insight and a next step
Write them in plain language. Then present for 2 minutes.

In [ ]:
top2 = summary.sort_values("pain_score", ascending=False).head(2)
insight = f'''
1. {top2.index[0]} and {top2.index[1]} create the most pain: many tickets, long resolution times and lower satisfaction.
2. {top2.index[0]} tickets take about {top2['avg_hours'].iloc[0]:.0f} hours on average to resolve.
3. Assumption to check: categories were inconsistently labelled ("slow app" vs "Slow app"), so labels need a quick review.
Next step: interview two support agents about {top2.index[0]} tickets this week and propose one fix by Friday.
'''
print(insight)

## 8. A decision log entry

In [ ]:
from datetime import date
decision_log = [{
    "date": str(date(2026, 11, 13)),
    "decision": f"Focus the first sprint on {top2.index[0]}",
    "options": f"{top2.index[0]}, {top2.index[1]}, Billing",
    "why": "Highest pain score; reversible if interviews disagree (two-way door)",
    "agreed_by": "Customer support lead (to confirm)",
}]
pd.DataFrame(decision_log)

### TODO: Write your own status update in the cell below using situation, action, next, ask.

In [ ]:
status_update = {
    "situation": "",
    "action": "",
    "next": "",
    "ask": "",
}
# TODO: fill in, then print it as four short lines

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)